# Time series

Working with time series data using `pandas`.

See more information about the data on the [course website](https://meds-eds-220.github.io/MEDS-eds-220-course/book/chapters/lesson-7-time-series.html).

## About the data

We will use hourly precipitation data from four weather stations in Boulder County, Colorado, from 2000 to 2013. In September 2013, an unusual weather pattern caused some of the most intense precipitation ever recorded in this region and led to [devastating floods](https://www.weather.gov/safety/flood-states-co). Our goal is to plot the 2013 precipitation data and find this event.

The data comes from the [NOAA Climate Data Online service](https://www.ncdc.noaa.gov/cdo-web/) ([full documentation](https://www.ncei.noaa.gov/pub/data/cdo/documentation/PRECIP_HLY_documentation.pdf)). These are the columns we will work with:

- `STATION`: ID number of the station.
- `STATION_NAME`: name of the station location.
- `DATE`: date and time of the record, written as `YYYYMMDD HH:MM`. Each value is the precipitation for the hour *ending* at this time.
- `HPCP`: precipitation in that hour, in inches. The value 999.99 means the data is missing. Hours with no precipitation are not in the data.
- `Measurement Flag`: notes about the record. `[` and `]` mark the start and end of a period with missing data, and `{` and `}` mark the start and end of a deleted period (data that was unreadable or clearly noise). Rows with these flags have an `HPCP` value of 999.99. `g` is only used for day 1 of the month at 01:00 when there was no precipitation, so these rows have an `HPCP` value of 0.

## Timestamps

`pandas` represents an instant in time using the `pandas.Timestamp` class:

In [ ]:
import pandas as pd

# Create a timestamp
pd.Timestamp(year=2020,
             month=10,
             day=18,
             hour=12,
             minute=30,
             second=15)

When we store several `pandas.Timestamp` objects in a `pandas.Series`, the data type of the series is `datetime64[ns]`:

In [ ]:
# Notice the data type of the series is datetime64
pd.Series([pd.Timestamp(2020, 10, 18),
           pd.Timestamp(2020, 10, 17),
           pd.Timestamp(2020, 10, 16)])

## Data exploration

Let's read in the data and take a look at it:

In [ ]:
# Read in data
URL = 'https://raw.githubusercontent.com/carmengg/eds-220-book/main/data/boulder_colorado_2013_hourly_precipitation.csv'
precip = pd.read_csv(URL)

precip

The data has records from more than one station. Let's check which ones using the `unique()` method:

There are four stations in Boulder County. The rows are ordered by station first and then by date: all the records from the first station (2000 to 2013), then all the records from the second station, and so on. In this lesson we will look at the total precipitation over the four stations.

Let's make a first plot of the data:

What is going on with this graph?


Let's fix these issues one at a time.

## Reading in missing data values

The metadata says that missing values are marked with 999.99. We can tell `pandas` this when we read in the data, using the `na_values` parameter of `pandas.read_csv()`:

The first `HPCP` value used to be 999.99 and now it is `NaN`. Check the `na_values` parameter in the [`pandas.read_csv()` documentation](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html) to see which values are read as `NA` by default.

Let's plot the data again:

## Converting strings into dates

The `DATE` column is not of type `datetime`. We can check this with the `dtypes` attribute:


We can convert strings into dates with the `pandas.to_datetime()` function:


#### Example

We can overwrite the `DATE` column with this output:

Let's plot the data again, now using the `DATE` column for the $x$-axis:

Notice the $x$-axis now shows years.

Next, we set the `DATE` column as the index. This lets us do operations based on time, like subsetting and resampling. We also sort the index using `sort_index()`:

By default, `plot()` uses the index for the $x$-axis and makes a line for each numeric column. So now we can simply call:

## Set the index when reading in the data

If we already know which column we want as the index, we can set it when we read in the data:


## Subsetting by date

`pandas` makes it easy to subset a data frame that has a time index:


#### Example

Select the data from September 2013:

Or select the data from a whole year:

We can plot this subset as usual. Notice the gaps due to missing data:

## Resample

Resampling means changing the frequency of a time series. For example, from monthly to yearly (downsampling) or from weekly to daily (upsampling). We can resample with the [`resample()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html) method:


`resample()` works similarly to `groupby()`: we need to say how to aggregate the data to get any output.

### Month end or month start?

`'ME'` stands for "month end" and `'MS'` for "month start". Let's compare them using the total monthly precipitation for the first three months of 2013:

In [ ]:
# Monthly totals labeled by the last day of each month
precip.loc['2013', 'HPCP'].resample('ME').sum().head(3)

In [ ]:
# Monthly totals labeled by the first day of each month
precip.loc['2013', 'HPCP'].resample('MS').sum().head(3)

#### Example

Our 2013 precipitation data is hourly. Let's resample it to daily:

To get an output, we need an aggregator function that says how to combine the data in each day. We want the total precipitation per day, so we use `sum()`:

The index is now the days in 2013. Since we selected a single column, the result is a `pandas.Series`. Let's rename it, since it is no longer hourly precipitation:

Finally, we can plot our data:

## Complete workflow

The code above includes a lot of exploration and trials. It is important to keep our data exploration documented. Once we are sure about our data wrangling, we can streamline our analysis to only include the code that makes the final output. This is the complete workflow for this lesson:

In [ ]:
import pandas as pd

# Read in hourly precipitation data from four stations in Boulder County, CO
# HPCP = hourly precipitation (unique numerical column in data frame)
URL = 'https://raw.githubusercontent.com/carmengg/eds-220-book/main/data/boulder_colorado_2013_hourly_precipitation.csv'
precip = (pd.read_csv(URL,
                      na_values={'HPCP': [999.99]},  # Known from metadata
                      index_col='DATE',
                      parse_dates=['DATE'])
          .sort_index()  # Rows in the file are ordered by station
          )


# Calculate daily total precipitation over all stations during 2013
daily_precip_2013 = (precip.loc['2013', 'HPCP']
                            .resample('D')
                            .sum()
                            .rename('daily_precipitation')
                            )

# Plot time series
daily_precip_2013.plot(ylabel='daily precipitation (in)',
                       xlabel='',
                       title='Total precipitation over four stations in Boulder County, CO (2013)')